# Machine Learning Workflow: Linear Regression Car Valuation
End-to-end pipeline: feature transformation, train-test splitting, fitting Linear Regression, performance metrics, and model saving via pickle.

In [ ]:
import os
import pickle
from datetime import datetime
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

### 1. Ingest Raw Dataset

In [ ]:
data_loc = os.path.join('..', 'data', 'raw data', 'car data.csv')
raw_cars = pd.read_csv(data_loc)
raw_cars.head()

### 2. Feature Cleaning & Indicator Encoding

In [ ]:
# Derive vehicle age from current calendar year
current_year = datetime.now().year
raw_cars['Age'] = current_year - raw_cars['Year']

# Remove raw identifiers
subset = raw_cars.drop(columns=['Car_Name', 'Year'])

# One-hot encode categorical features (drop first level to prevent collinearity)
processed_cars = pd.get_dummies(subset, drop_first=True, dtype=int)

# Consistent feature ordering
ordered_cols = ['Selling_Price', 'Present_Price', 'Kms_Driven', 'Owner', 'Age',
                'Fuel_Type_Diesel', 'Fuel_Type_Petrol', 'Seller_Type_Individual', 'Transmission_Manual']
processed_cars = processed_cars[ordered_cols]
processed_cars.head()

### 3. Save Processed Dataset to Disk

In [ ]:
target_processed_csv = os.path.join('..', 'data', 'processed data', 'processed_data.csv')
processed_cars.to_csv(target_processed_csv, index=False)
print(f'Processed dataset successfully written to: {target_processed_csv}')

### 4. Partition Train and Test Sets

In [ ]:
X = processed_cars.drop(columns=['Selling_Price'])
y = processed_cars['Selling_Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f'Train samples: {X_train.shape[0]} | Test samples: {X_test.shape[0]}')

### 5. Train Linear Regression Model

In [ ]:
reg_model = LinearRegression()
reg_model.fit(X_train, y_train)
print('Training complete.')

### 6. Model Evaluation

In [ ]:
y_pred = reg_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'R2 Score (Variance Explained): {r2:.4f}')
print(f'Mean Absolute Error (MAE)    : {mae:.4f} Lakhs')
print(f'Root Mean Squared Error (RMSE): {rmse:.4f} Lakhs')

### 7. Fitted Weights (Feature Impact)

In [ ]:
weights_df = pd.DataFrame({
    'Feature_Name': X.columns,
    'Weight_Value': reg_model.coef_
}).sort_values(by='Weight_Value', ascending=False)
weights_df

### 8. Pickle Serialization & Verification

In [ ]:
model_export_path = os.path.join('..', 'models', 'lr_model.pkl')
with open(model_export_path, 'wb') as outfile:
    pickle.dump(reg_model, outfile)
print(f'Model persisted to: {model_export_path}')

# Verify reload
with open(model_export_path, 'rb') as infile:
    loaded_reg = pickle.load(infile)
sample_pred = loaded_reg.predict(X_test.iloc[:1])
print(f'Sample Test Prediction: {sample_pred[0]:.2f} Lakhs (Actual Value: {y_test.iloc[0]:.2f} Lakhs)')